# Random Semantic Algebra — Large-Scale Paper Benchmark

Public-repo Colab. No GitHub token is required.

- Smoke: 8k products, 20 compound queries, 1 seed
- Full: all ~44k products, 200 queries × 3 seeds
- MiniLM retrieval, CLIP independent teacher
- Dense-only vs RSA vs FP32 semantic proxy vs Oracle


In [ ]:
#@title 1) Choose run mode
FULL_RUN = False #@param {type:"boolean"}
RUN_TESTS = True #@param {type:"boolean"}
print("FULL_RUN =", FULL_RUN)


In [ ]:
#@title 2) Clone public repo and install
import os, subprocess, pathlib, shutil
ROOT=pathlib.Path('/content/ras')
if ROOT.exists(): shutil.rmtree(ROOT)
subprocess.run(['git','clone','--depth=1','https://github.com/hanialshater/ras.git',str(ROOT)],check=True)
os.chdir(ROOT)
print('repo:', subprocess.check_output(['git','rev-parse','--short','HEAD']).decode().strip())
subprocess.run(['pip','install','-q','-e','.'],check=True)
print('installed')


In [ ]:
#@title 3) Tests
import subprocess, os
os.chdir('/content/ras')
if RUN_TESTS:
    subprocess.run(['pytest','-q'],check=True)
else:
    print('tests skipped')


In [ ]:
#@title 4) Run benchmark
import os, subprocess, time
os.chdir('/content/ras')
config='configs/large_scale.yaml' if FULL_RUN else 'configs/smoke.yaml'
print('config:',config)
t0=time.time()
subprocess.run(['python','-m','experiments.large_scale_search','--config',config],check=True)
print(f'finished in {(time.time()-t0)/60:.1f} min')


In [ ]:
#@title 5) Show latest results
from pathlib import Path
import json, pandas as pd
root=Path('/content/ras/results')
runs=sorted([p for p in root.iterdir() if p.is_dir()],key=lambda p:p.stat().st_mtime)
run=runs[-1]
print('latest run:',run)
print(json.dumps(json.loads((run/'headline.json').read_text()),indent=2))
display(pd.read_csv(run/'summary.csv'))
display(pd.read_csv(run/'paired_deltas.csv'))
display(pd.read_csv(run/'predicate_metrics.csv'))


In [ ]:
#@title 6) Show figures
from IPython.display import display, Image
for p in sorted((run/'figures').glob('*.png')):
    print(p.name)
    display(Image(filename=str(p)))


In [ ]:
#@title 7) Zip results
import shutil
zip_path=shutil.make_archive(f'/content/{run.name}','zip',root_dir=str(run))
print(zip_path)
